# 02 — Pre-train the masked financial-event model

The objective is bidirectional masked value modelling. Inputs are structured transaction key/value pairs; selected value IDs are hidden and the model predicts their original vocabulary IDs. Local event context, the materialized profile state, and contextual event history all lie on the path to the MLM loss.

This notebook deliberately keeps the implementation in `pragma_lite.training.run_pretraining`. The notebook configures a run, explains it, and stores durable artifacts.

## Colab setup

A VS Code–Colab connection runs this kernel on a temporary Colab VM. Keep the repository and Parquet reads under fast `/content`; Drive is for persistence only. If this notebook is opened outside an existing `/content/FinancialBertForTransactions` clone, clone the repository first as shown in the README, then run this cell.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    candidate = Path('/content/FinancialBertForTransactions')
    if candidate.exists():
        PROJECT_ROOT = candidate
    else:
        raise RuntimeError('Clone the repository under /content, then rerun this cell.')
SOURCE_DIR = PROJECT_ROOT / 'src'
if not SOURCE_DIR.is_dir():
    raise RuntimeError(f'Could not find package source directory: {SOURCE_DIR}')
# Colab already provides Jupyter. Install only this project's runtime
# dependencies here so we do not replace Colab's own kernel packages.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=PROJECT_ROOT, check=True)
# The editable install is the durable setup. Adding src here also makes the
# current Jupyter process robust when VS Code attaches a kernel whose import
# paths were created before pip ran.
os.chdir(PROJECT_ROOT)
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
import pragma_lite
print(f'Using kernel: {sys.executable}')
print(f'Using package: {Path(pragma_lite.__file__).resolve()}')
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / 'czech_bank'
assert (PROCESSED_DIR / 'events_train.parquet').exists()

## Mount Drive only for checkpoints

The model does not train from Drive. Writing frequent batch I/O there is slower and adds avoidable instability. This mount preserves `best.pt`, fitted tokenizer states, and the concise training report across a 12-hour Colab session boundary.

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/FinancialBertForTransactions/checkpoints/pragma_lite_mlm')
else:
    CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / 'pragma_lite_mlm'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR

## Run configuration

The initial portfolio run uses 128 hidden dimensions, eight heads, two blocks per encoder, and a 256-event context. This is intentionally small enough to iterate within Colab sessions while still exercising the complete Event → Profile → History path. The model has under one million parameters.

An earlier run showed a rapid fall in validation MLM loss during its opening epochs, so the pipeline is learning a nontrivial structured-token distribution. Loss alone is not the portfolio claim; the downstream notebook tests whether the account embeddings transfer to cutoff-safe tasks.

In [ ]:
from pragma_lite.models import TransformerConfig
from pragma_lite.training import PretrainingConfig, run_pretraining

CONFIG = PretrainingConfig(
    model=TransformerConfig(d_model=128, num_heads=8, num_layers=2, ffn_dim=256, dropout=0.1),
    max_events=256,
    batch_size=128,
    epochs=100,
    learning_rate=3e-4,
)
RESUME = False  # True resumes CHECKPOINT_DIR / 'last.pt' at the next epoch.
report = run_pretraining(processed_dir=PROCESSED_DIR, checkpoint_dir=CHECKPOINT_DIR, config=CONFIG, resume=RESUME)
report['checkpoint'], report['best_validation_loss']

`best.pt` contains the model weights, model configuration, maximum context length, and the information needed to recreate the backbone. The adjacent `tokenizers/` directory contains only tokenizer state fitted from training records. Continue to notebook 03 with this checkpoint path.